# CNMFe parameter tuning — interactive viewer

A **thin viewer** over the `tuning/` package: every cell calls into
`tuning.heuristics` / `tuning.report` / `tuning.tuner` — no tuning logic lives
here. For a headless run use the CLI instead:

```bash
python tune.py /path/to/avis -o tuning/ --frame-rate 20 --decay-time-ms 180 --mode both
```

Two ways to use this notebook:
- **Automated** (section 1): fill in `TunerConfig`, run `run_tuning`, open the
  report folder it writes.
- **Interactive** (section 2): step through each stage and display the same
  figures inline so you can tweak inputs and re-judge.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/home/fs539/code/simpler_cnmfe')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from minicnmfe.io import open_zarr
from tuning import heuristics as H, io_sample as S, report as R
from tuning.sweep import SweepSpec
from tuning.tuner import TunerConfig, run_tuning
print('imports ok')

## 1. Automated run

Edit the config, then run. Point `input_path` at an AVI folder or an `mc.zarr`.

In [ ]:
# --- EDIT ME ---------------------------------------------------------------
INPUT = Path('/path/to/miniscope_video')      # AVI folder OR an mc.zarr
OUTPUT = Path('/tmp/cnmfe_tuning')             # parent for the run folder
FRAME_RATE_HZ = 20.0
DECAY_TIME_MS = 180.0                           # GCaMP8m ~180, 8f ~70, 6f ~140
MODE = 'both'                                   # 'heuristic' | 'sweep' | 'both'
REGION = 'cutout'                               # 'cutout' | 'full'
# ---------------------------------------------------------------------------

cfg = TunerConfig(
    input_path=INPUT, output_dir=OUTPUT / 'tune_interactive',
    mode=MODE, region=REGION,
    frame_rate_hz=FRAME_RATE_HZ, decay_time_ms=DECAY_TIME_MS,
    sweep=SweepSpec(min_corr=[0.7, 0.8], min_pnr=[6.0, 10.0],
                    global_bg_rank=[0, 1]),
    n_jobs=-1,
)
result = run_tuning(cfg)
print('report folder:', cfg.output_dir)
print('recommended:', result['recommended'])

## 2. Interactive stage-by-stage

The same functions, displayed inline. Each `report.fig_*(evidence)` returns a
Matplotlib figure when no `out_path` is given.

### 2.1 Motion-correction heuristics (AVI input)

In [ ]:
avis = S.list_avis(INPUT)                       # requires INPUT to be an AVI folder
sample = S.decode_strided_sample(avis, n_avis=8, stride=50)
median_img = np.median(sample, axis=0)

mc_gsig, sigma_native, ev = H.suggest_mc_gsig_and_sigma(sample)
print('mc_gSig_filt =', mc_gsig, ' sigma_native =', round(sigma_native, 2))
R.fig_mc_gsig(ev); plt.show()

max_shift, border_px, ev = H.suggest_max_shift(sample, median_img, mc_gsig)
print('max_shift =', max_shift, ' border_px =', border_px)
R.fig_max_shift(ev); plt.show()

ssub, tsub, ev = H.suggest_downsample(sigma_native, FRAME_RATE_HZ, DECAY_TIME_MS)
print('ssub =', ssub, ' tsub =', tsub)
R.fig_downsample(ev); plt.show()

### 2.2 Initialisation heuristics (on an mc.zarr)

Use `cfg.output_dir/mc/mc.zarr` from the automated run, or any existing `mc.zarr`.

In [ ]:
MC_ZARR = cfg.output_dir / 'mc' / 'mc.zarr'     # or an existing mc.zarr path
mc = open_zarr(MC_ZARR)
mc_sample, idx = S.load_mc_sample(mc, n_frames=400)
dims = (mc.shape[1], mc.shape[2])

sigma_ds, cn, pnr, ev = H.suggest_sigma_extraction(mc_sample, max(2.0, sigma_native))
print('sigma (extraction grid) =', round(sigma_ds, 2))
R.fig_corr_pnr_sigma(ev); plt.show()

min_corr, min_pnr, ev = H.suggest_corr_pnr(cn, pnr, sigma_ds)
print('min_corr =', min_corr, ' min_pnr =', min_pnr)
R.fig_seed_heatmap(ev); plt.show()

min_pixel, ev = H.suggest_min_pixel(mc_sample, sigma_ds, min_corr, min_pnr, dims)
print('min_pixel =', min_pixel)
R.fig_min_pixel(ev); plt.show()

### 2.3 Temporal / merge / eval heuristics

These need a fitted model — either the sweep-best (`run_tuning` returns it via
the report), or load a previous `results/` dir with `CNMFe.load(...)`.

In [ ]:
from minicnmfe.pipeline import CNMFe
model = CNMFe.load('/path/to/results')          # a previous fit_extract result
if model.eval_info is None and model.sn is not None:
    model.evaluate()

decay_ms, ev = H.suggest_decay_time(model, FRAME_RATE_HZ)
print('decay_time_ms =', decay_ms)
R.fig_decay(ev); plt.show()

gpw, ev = H.suggest_g_prior_weight(ev['g_yw'], FRAME_RATE_HZ, decay_ms)
print('g_prior_weight =', gpw)
R.fig_g_prior(ev); plt.show()

merge_thr, ev = H.suggest_merge_thr(model);  print('merge_thr_corr =', merge_thr)
R.fig_merge_corr(ev); plt.show()

snr_thr, ev = H.suggest_snr_thr(model);  print('auto_eval_snr_amp_thr =', snr_thr)
R.fig_snr_eval(ev, model); plt.show()